## Time Travel from Spark

In [7]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [8]:
conf = SparkConf()

conf.setAppName("Delta Table History") # nome da aplicação Spark, útil para logs
conf.set("spark.hadoop.fs.s3a.endpoint","http://minio:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", "admin") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", "@admin123") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True)
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

## reading file from MinIO S3 bronze bucket in Delta format

In [3]:
dataFrame = spark.read.format("delta").load("s3a://bronze/delta_example")

In [4]:
dataFrame.show()

+----------+---------+------+------+
|first_name|last_name|Gender|salary|
+----------+---------+------+------+
|   Rodrigo| Maturino|     M|  4000|
|   Eduarda|   Santos|     F|  5000|
|    Rafael|    Souza|     M|  4500|
|     Luigi|    Bispo|     M|  6000|
+----------+---------+------+------+



## Adding a Delta table in MinIO S3 bronze bucket

In [12]:
dataFrame.write.format("delta").mode("append").save("s3a://bronze/time_travel_example")

## Analyze number version table

In [14]:
table_path = "s3a://bronze/time_travel_example"
delta_table = DeltaTable.forPath(spark, table_path)
history_df = delta_table.history()
history_df.show(truncate=False)

+-------+-------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|2      |2026-04-23 23:15:14|null  |null    |WRITE    |{mode -> Append, partitionBy -> []}|null|null    |null     |1          |Serializable  |true         |{numFiles -> 4, numOutputRows -> 4, numOutputB

## Adding a new version in the table

In [10]:
dataFrame.write.format("delta").mode("append").save("s3a://bronze/time_travel_example")

## Show the new Version

In [13]:
table_path = "s3a://bronze/time_travel_example"
delta_table = DeltaTable.forPath(spark, table_path)
history_df = delta_table.history()
history_df.show(truncate=False)

+-------+-------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|0      |2026-04-23 23:12:17|null  |null    |WRITE    |{mode -> Append, partitionBy -> []}|null|null    |null     |null       |Serializable  |true         |{numFiles -> 4, numOutputRows -> 4, numOutputB

## Read the first Version

In [16]:
spark.read.format("delta").option("versionAsOf", 0).load(table_path).show()

+----------+---------+------+------+
|first_name|last_name|Gender|salary|
+----------+---------+------+------+
|   Rodrigo| Maturino|     M|  4000|
|   Rodrigo| Maturino|     M|  4000|
|   Eduarda|   Santos|     F|  5000|
|   Eduarda|   Santos|     F|  5000|
|    Rafael|    Souza|     M|  4500|
|    Rafael|    Souza|     M|  4500|
|     Luigi|    Bispo|     M|  6000|
|     Luigi|    Bispo|     M|  6000|
+----------+---------+------+------+



## Rollback to version 0

In [18]:
df_history_0 = spark.read.format("delta").option("versionAsOf", 0).load(table_path)
df_history_0.write.format("delta").mode("overwrite").save(table_path)

## Read table in actual version

In [19]:
df_actual_version = spark.read.format("delta").load(table_path).show()

+----------+---------+------+------+
|first_name|last_name|Gender|salary|
+----------+---------+------+------+
|   Rodrigo| Maturino|     M|  4000|
|   Eduarda|   Santos|     F|  5000|
|    Rafael|    Souza|     M|  4500|
|     Luigi|    Bispo|     M|  6000|
+----------+---------+------+------+

